# पाठ ०८ - बहु-एजेन्ट डिजाइन ढाँचा


## सेटअप


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## किन बहु-एजेन्ट प्रणालीहरू?

वास्तविक जीवनका कार्यहरू जस्तै यात्रा योजना बनाउन विभिन्न प्रकारका विशेषज्ञता आवश्यक पर्दछ — ढुवानी, स्थानीय ज्ञान, बजेटिङ, र थप। सबै कुरा सम्हाल्न खोज्ने एकल एजेन्ट छिटो असहज हुन्छ।

बहु-एजेन्ट प्रणालीहरूले यसलाई **विशेषज्ञता** मार्फत समाधान गर्छन्: प्रत्येक एजेन्टले एउटा विशिष्ट क्षेत्रमा ध्यान केन्द्रित गर्छ, जसले सामान्यजनको तुलनामा उच्च गुणस्तरको परिणाम उत्पादन गर्छ। यसले **विस्तारयोग्यता** पनि सुधार्छ — तपाईं नयाँ एजेन्टहरू थप्न सक्नुहुन्छ (जस्तै, उडान विशेषज्ञ, रेस्टुरेन्ट समीक्षक) बिना हालको कार्यप्रवाह पुनःलेखन। एजेन्टहरू एक संरचित पाइपलाइन मार्फत सँगै मिलेर काम गर्छन्, एउटा एजेन्टबाट अर्कोमा सन्दर्भ पास गर्दै।


## विशेष एजेन्टहरू सिर्जना गर्दैछौं


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## क्रमिक कार्यप्रवाह बनाउँदै

`WorkflowBuilder` ले तपाईंलाई एजेन्टहरूलाई निर्देशित ग्राफमा तार जडान गर्न दिन्छ। यहाँ हामी एक सरल दुई-चरण पाइपलाइन बनाउँछौं: **TravelPlanner** ले यात्रा तालिका तयार पार्छ, त्यसपछि **TravelConcierge** ले त्यसलाई समीक्षा गरी सुधार गर्छ।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## वर्कफ्लोमा थप एजेन्टहरू थप्दै

मल्टि-एजेन्ट ढाँचाको सबैभन्दा ठूलो फाइदाहरू मध्ये एक हो कति सजिलै यसलाई विस्तार गर्न सकिन्छ। तल हामीले एक **BudgetReviewer** एजेन्ट थपेका छौं जसले यात्रु को बजेटको विरुद्ध योजनाको जाँच गर्छ, लागत सीमा पार गर्ने वस्तुहरूलाई झण्डा लगाउँछ, र पैसा बचत गर्ने विकल्पहरू सुझाव दिन्छ। वर्कफ्लो अब तीन एजेन्टहरूलाई क्रमशः चलाउँछ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## सारांश

यस पाठमा तपाईंले कसरी गर्ने जान्नुभयो:

1. **विशेषज्ञ एजेन्टहरू सिर्जना गर्ने** — प्रत्येकले एक केन्द्रित भूमिका (योजना, कन्सियर्ज, बजेट समीक्षा)।
2. **एजेन्टहरूलाई क्रमिक कार्यप्रवाहमा जडान गर्ने** `WorkflowBuilder` र `add_edge` प्रयोग गरेर।
3. **धेरै-एजेन्ट पाइपलाइनबाट आउटपुट स्ट्रिम गर्ने**, कुन एजेन्ट बोलिरहेको छ ट्र्याक गर्दै।
4. **कार्यप्रवाह विस्तार गर्ने** नयाँ एजेन्टहरू श्रृंखलामा थपेर, विद्यमानहरूलाई परिवर्तन नगरी।

धेरै-एजेन्ट डिजाइन ढाँचाले प्रत्येक एजेन्टलाई सरल राख्छ जबकि कुनै एकल एजेन्टले एक्लैले पुरा गर्न सक्नेभन्दा धनी, थप सावधानीपूर्वक समीक्षा गरिएका परिणामहरू उत्पादन गर्छ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अपवाद**:
यस दस्तावेजलाई AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी शुद्धताको प्रयास गर्छौं भने पनि, कृपया जानकार हुनुस् कि स्वतःअनुवादमा त्रुटि वा अस्वीकार्यताहरू हुनसक्छन्। मूल दस्तावेज आफ्नो मूल भाषामा नै अधिकारिक स्रोत मान्नुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानवीय अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न हुने कुनै पनि गलतफहमी वा गलत व्याख्यामा हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
